# 08 — NCCS Historical Data Integration (2010–2024) 

Combines the original Notebook 08's download, inventory, and feature engineering with the newer Notebook 08's disclosure modeling and investigation queue. **Run top to bottom only when intentionally recomputing the analysis.** All saved outputs were cleared so the notebook is reproducible.

**Goal:** Rank current-year nonprofit filings for investigative review, not establish fraud. The label represents a *self-reported significant diversion of assets*. Model performance on that label does not validate undisclosed misconduct.

**Before running:** Select the `irs990` Python kernel. Ensure `duckdb`, `requests`, `pyarrow`, `pandas`, `numpy`, and `scikit-learn` are installed. Data is intentionally not tracked by Git.

### Running This Notebook

This notebook integrates historical NCCS data, engineers financial features, and trains models.

**Default settings (safe mode):**
- `RUN_DOWNLOADS = False` — Uses existing NCCS source files without downloading or replacing them.
- `WRITE_OUTPUTS = False` — Runs the analysis without regenerating the three protected Parquet exports.

**To regenerate results:**
1. Set `RUN_DOWNLOADS = True` if the source data needs to be downloaded.
2. Set `WRITE_OUTPUTS = True` to save regenerated datasets.
3. Run the notebook from the beginning.

**Important:** Regenerating outputs may overwrite existing files. Preserve the original datasets and model artifacts before rerunning.

## 1. Resolve project paths and configure sources

In [ ]:
from pathlib import Path
import pandas as pd
import duckdb
import pyarrow.parquet as pq
import requests

RUN_DOWNLOADS = False
WRITE_OUTPUTS = False

# Works when VS Code starts in either the repository root or its notebooks directory.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'notebooks').is_dir() and (p / 'environment.yml').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open the irs-990-fraud-detection project folder in VS Code before running.')
NCCS_DIR = PROJECT_ROOT / 'data/external/nccs/efile_v3_1'
OUTPUT_DIR = PROJECT_ROOT / 'data/processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NCCS_DIR.mkdir(parents=True, exist_ok=True)
YEARS = range(2010, 2025)
TABLES = {'header':'F9-P00-T00-HEADER','summary':'F9-P01-T00-SUMMARY','governance':'F9-P06-T00-GOVERNANCE'}
BASE_URL = 'https://nccs-efile.s3.us-east-1.amazonaws.com/public/efile_v3_1'
print('Project root:', PROJECT_ROOT)
print('NCCS folder:', NCCS_DIR)


## 2. Download or validate the 45 source Parquet files

Downloads are disabled by default (`RUN_DOWNLOADS = False`) to protect existing source files.

To download missing or invalid source files, explicitly set `RUN_DOWNLOADS = True` in Section 1. Existing valid files are skipped. Files that fail validation may be replaced when downloads are enabled.

Downloading the full source dataset may require substantial storage and bandwidth.

In [ ]:
if RUN_DOWNLOADS:
    downloaded, skipped, failed = [], [], []

    for year in YEARS:
        folder = NCCS_DIR / str(year)
        folder.mkdir(parents=True, exist_ok=True)

        for prefix in TABLES.values():
            filename = f'{prefix}-{year}.parquet'
            destination = folder / filename

            if destination.exists():
                try:
                    pq.ParquetFile(destination)
                    skipped.append(filename)
                    continue
                except Exception:
                    destination.unlink()

            temporary = destination.with_suffix('.parquet.part')

            try:
                with requests.get(
                    f'{BASE_URL}/{filename}',
                    stream=True,
                    timeout=(30, 180)
                ) as response:
                    response.raise_for_status()

                    with temporary.open('wb') as handle:
                        for chunk in response.iter_content(chunk_size=1024*1024):
                            if chunk:
                                handle.write(chunk)

                pq.ParquetFile(temporary)
                temporary.replace(destination)
                downloaded.append(filename)
                print('Downloaded:', filename)

            except Exception as exc:
                failed.append((filename, str(exc)))
                temporary.unlink(missing_ok=True)
                print('FAILED:', filename, exc)

    print(
        'Downloaded:', len(downloaded),
        'Skipped:', len(skipped),
        'Failed:', len(failed)
    )

    if failed:
        raise RuntimeError(
            'Some NCCS downloads failed; retry this cell before continuing.'
        )

else:
    print("Downloads disabled. Existing NCCS files will be used.")

In [ ]:
from pathlib import Path

base = PROJECT_ROOT / "data/external/nccs/efile_v3_1"

files = sorted(base.rglob("*.parquet"))

print("NCCS folder:", base)
print("Total Parquet files:", len(files))

for year in range(2010, 2025):
    year_files = [f for f in files if f.parent.name == str(year)]
    print(f"{year}: {len(year_files)} files")

## 3. Verify complete 2010–2024 coverage

In [ ]:
inventory=[]
for year in YEARS:
    for table,prefix in TABLES.items():
        path=NCCS_DIR / str(year) / f'{prefix}-{year}.parquet'
        inventory.append({'year':year,'table':table,'exists':path.is_file(),
                          'rows':pq.ParquetFile(path).metadata.num_rows if path.is_file() else None})
inventory_df=pd.DataFrame(inventory)
display(inventory_df.pivot(index='year',columns='table',values='rows'))
assert len(inventory_df)==45 and inventory_df['exists'].all(), 'Missing expected NCCS Parquet files.'
print('All 45 files found; physical row counts shown above.')


## 4. Register DuckDB views using absolute file paths

This fixes the original `No files found that match the pattern` error when the notebook runs from a different working directory.

In [ ]:
con = duckdb.connect(database=':memory:')
con.execute('PRAGMA threads=4')
con.execute("SET memory_limit='8GB'")
con.execute(f"SET temp_directory='{(PROJECT_ROOT / 'data/duckdb_tmp').as_posix()}'")
for table,prefix in TABLES.items():
    glob=(NCCS_DIR / '*' / f'{prefix}-*.parquet').as_posix()
    con.execute(f"CREATE OR REPLACE VIEW nccs_{table} AS SELECT * FROM read_parquet('{glob}', union_by_name=true)")
    print(table, 'records:',con.execute(f'SELECT COUNT(*) FROM nccs_{table}').fetchone()[0])


## 5. Inspect source keys and confirm join cardinality

In [ ]:
key_audit=[]
for table in TABLES:
    key_audit.append(con.execute(f"""SELECT '{table}' AS source, COUNT(*) AS rows,
        COUNT(*) FILTER (WHERE OBJECTID IS NULL) AS null_objectids,
        COUNT(DISTINCT OBJECTID) AS distinct_objectids
        FROM nccs_{table}""").df())
key_audit_df=pd.concat(key_audit,ignore_index=True)
display(key_audit_df)
if (key_audit_df.null_objectids > 0).any() or (key_audit_df.rows != key_audit_df.distinct_objectids).any():
    raise ValueError('OBJECTID is not unique/non-null in at least one source. Resolve joins before continuing; run Notebook 08A.')


## 6. Create filing-level historical master with explicit label missingness

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE nccs_master_historical AS
SELECT h.OBJECTID, TRY_CAST(h.tax_year AS INTEGER) AS tax_year,
       CAST(h.EIN2 AS VARCHAR) AS ein,
       s.F9_01_NAFB_ASSET_TOT_EOY AS total_assets,
       s.F9_01_NAFB_LIAB_TOT_EOY AS total_liabilities,
       s.F9_01_REV_TOT_CY AS total_revenue,
       s.F9_01_EXP_TOT_CY AS total_expenses,
       g.F9_06_GVRN_NUM_VOTING_MEMB AS voting_members,
       g.F9_06_GVRN_NUM_VOTING_MEMB_IND AS independent_members,
       CAST(g.F9_06_GVRN_ASSET_DIVERSION_X AS VARCHAR) AS raw_diversion_disclosure,
       CASE
         WHEN UPPER(TRIM(CAST(g.F9_06_GVRN_ASSET_DIVERSION_X AS VARCHAR))) IN ('1','Y','YES','TRUE') THEN 1
         WHEN UPPER(TRIM(CAST(g.F9_06_GVRN_ASSET_DIVERSION_X AS VARCHAR))) IN ('0','N','NO','FALSE') THEN 0
         ELSE NULL
       END AS fraud_label,
       s.OBJECTID IS NOT NULL AS has_summary,
       g.OBJECTID IS NOT NULL AS has_governance
FROM nccs_header h
LEFT JOIN nccs_summary s ON h.OBJECTID = s.OBJECTID
LEFT JOIN nccs_governance g ON h.OBJECTID = g.OBJECTID
""")
master_count=con.execute('SELECT COUNT(*) FROM nccs_master_historical').fetchone()[0]
header_count=con.execute('SELECT COUNT(*) FROM nccs_header').fetchone()[0]
assert master_count==header_count, f'Join multiplication detected: {master_count} vs {header_count}'
display(con.execute("""SELECT tax_year, COUNT(*) AS filings,
 SUM(CAST(has_summary AS INTEGER)) AS with_summary,
 SUM(CAST(has_governance AS INTEGER)) AS with_governance,
 COUNT(*) FILTER (WHERE fraud_label=1) AS positives,
 COUNT(*) FILTER (WHERE fraud_label IS NULL) AS unknown_labels
 FROM nccs_master_historical GROUP BY 1 ORDER BY 1""").df())
print('Raw disclosure values (inspect before modeling):')
display(con.execute("""SELECT raw_diversion_disclosure, COUNT(*) n FROM nccs_master_historical GROUP BY 1 ORDER BY n DESC LIMIT 25""").df())


## 7. Audit EIN/year duplicates and choose a canonical filing

Selects the latest available filing for each EIN and tax year using the return timestamp, amendment flag, and OBJECTID as a deterministic tie-breaker.

In [ ]:
# Audit duplicate filings by organization and tax year
display(con.execute("""
    SELECT
        tax_year,
        COUNT(*) AS filings,
        COUNT(DISTINCT ein) AS unique_eins,
        COUNT(*) - COUNT(DISTINCT ein) AS additional_filings
    FROM nccs_master_historical
    GROUP BY tax_year
    ORDER BY tax_year
""").df())

# Select the latest filing for each organization and tax year
con.execute("""
    CREATE OR REPLACE TABLE nccs_master_dedup AS
    WITH ranked AS (
        SELECT
            m.*,
            ROW_NUMBER() OVER (
                PARTITION BY m.ein, m.tax_year
                ORDER BY
                    TRY_CAST(
                        h.RETURN_TIME_STAMP AS TIMESTAMPTZ
                    ) DESC NULLS LAST,
                    CASE
                        WHEN UPPER(TRIM(h.RETURN_AMENDED_X)) = 'TRUE'
                        THEN 1
                        ELSE 0
                    END DESC,
                    m.OBJECTID DESC
            ) AS rn
        FROM nccs_master_historical m
        LEFT JOIN nccs_header h
            ON m.OBJECTID = h.OBJECTID
        WHERE m.ein IS NOT NULL
          AND m.tax_year IS NOT NULL
    )
    SELECT * EXCLUDE (rn)
    FROM ranked
    WHERE rn = 1
""")

print(
    "Deduplicated filings:",
    con.execute(
        "SELECT COUNT(*) FROM nccs_master_dedup"
    ).fetchone()[0]
)

display(con.execute("""
    SELECT
        COUNT(*) AS deduplicated_filings,
        COUNT(DISTINCT ein) AS unique_organizations
    FROM nccs_master_dedup
""").df())



## 8. Feature engineering, longitudinal trends, and exports

In [ ]:
from pathlib import Path

# Ensure export directory exists
output_dir = OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)

# Build normalized feature matrix with financial ratios & governance flags
con.execute("""
    CREATE OR REPLACE TABLE nccs_feature_matrix_v1 AS
    SELECT 
        OBJECTID,
        tax_year,
        ein,
        fraud_label,
        
        -- Numeric Financial Features
        TRY_CAST(total_assets AS DOUBLE) AS total_assets,
        TRY_CAST(total_liabilities AS DOUBLE) AS total_liabilities,
        TRY_CAST(total_revenue AS DOUBLE) AS total_revenue,
        TRY_CAST(total_expenses AS DOUBLE) AS total_expenses,
        
        -- Financial Ratios
        (TRY_CAST(total_revenue AS DOUBLE) - TRY_CAST(total_expenses AS DOUBLE)) AS net_income,
        CASE 
            WHEN TRY_CAST(total_assets AS DOUBLE) > 0 
            THEN (TRY_CAST(total_liabilities AS DOUBLE) / TRY_CAST(total_assets AS DOUBLE))
            ELSE NULL 
        END AS debt_to_asset_ratio,
        
        -- Governance Features
        TRY_CAST(voting_members AS DOUBLE) AS voting_members,
        TRY_CAST(independent_members AS DOUBLE) AS independent_members,
        
        -- Engineered Governance Risk Flags
        CASE
            WHEN TRY_CAST(voting_members AS DOUBLE) IS NULL THEN NULL
            WHEN TRY_CAST(voting_members AS DOUBLE) < 3 THEN 1
            ELSE 0
        END AS flag_small_board,
        CASE 
            WHEN TRY_CAST(voting_members AS DOUBLE) > 0 
            THEN (TRY_CAST(independent_members AS DOUBLE) / TRY_CAST(voting_members AS DOUBLE))
            ELSE NULL 
        END AS pct_independent_board,
        CASE
            WHEN TRY_CAST(voting_members AS DOUBLE) > 0
             AND TRY_CAST(independent_members AS DOUBLE) IS NOT NULL
            THEN CASE
                WHEN TRY_CAST(independent_members AS DOUBLE)
                     / TRY_CAST(voting_members AS DOUBLE) < 0.5
                THEN 1 ELSE 0
            END
            ELSE NULL
        END AS flag_minority_independent_board

    FROM nccs_master_dedup;
""")

# Export feature matrix Parquet
feature_export_path = output_dir / "nccs_feature_matrix_v1.parquet"
if WRITE_OUTPUTS:
    con.execute(f"""
        COPY nccs_feature_matrix_v1
        TO '{feature_export_path}'
        (FORMAT PARQUET, COMPRESSION 'SNAPPY');
    """)
    print(f"Feature matrix saved to: {feature_export_path.resolve()}")
else:
    print("Feature matrix export skipped.")

In [ ]:
# Compute multi-year rolling trend & volatility features carrying forward all columns (f.*)

con.execute("""
    CREATE OR REPLACE TABLE nccs_longitudinal_features_v1 AS
    WITH lagged AS (
        SELECT
            f.*,

            LAG(tax_year, 1) OVER (
                PARTITION BY ein ORDER BY tax_year
            ) AS prior_year,

            LAG(tax_year, 2) OVER (
                PARTITION BY ein ORDER BY tax_year
            ) AS two_years_prior,

            LAG(total_revenue, 1) OVER (
                PARTITION BY ein ORDER BY tax_year
            ) AS rev_lag1,

            LAG(total_revenue, 2) OVER (
                PARTITION BY ein ORDER BY tax_year
            ) AS rev_lag2,

            LAG(total_assets, 1) OVER (
                PARTITION BY ein ORDER BY tax_year
            ) AS assets_lag1,

            LAG(total_assets, 2) OVER (
                PARTITION BY ein ORDER BY tax_year
            ) AS assets_lag2,

            LAG(net_income, 1) OVER (
                PARTITION BY ein ORDER BY tax_year
            ) AS net_inc_lag1,

            LAG(pct_independent_board, 1) OVER (
                PARTITION BY ein ORDER BY tax_year
            ) AS indep_board_lag1

        FROM nccs_feature_matrix_v1 f
    ),
    validated AS (
        SELECT
            *,

            CASE
                WHEN prior_year = tax_year - 1
                 AND two_years_prior = tax_year - 2
                THEN 1 ELSE 0
            END AS has_three_consecutive_years

        FROM lagged
    )
    SELECT
        *,

        CASE
            WHEN has_three_consecutive_years = 1
             AND rev_lag2 > 0
            THEN (total_revenue - rev_lag2) / rev_lag2
            ELSE NULL
        END AS rev_growth_3yr,

        CASE
            WHEN has_three_consecutive_years = 1
             AND assets_lag2 > 0
            THEN (total_assets - assets_lag2) / assets_lag2
            ELSE NULL
        END AS asset_growth_3yr,

       CASE
            WHEN prior_year <> tax_year - 1
              OR prior_year IS NULL
              OR net_income IS NULL
              OR net_inc_lag1 IS NULL
            THEN NULL
            WHEN net_income < 0
             AND net_inc_lag1 < 0
            THEN 1
            ELSE 0
        END AS flag_sustained_deficit,

        CASE
            WHEN prior_year <> tax_year - 1
              OR prior_year IS NULL
              OR indep_board_lag1 IS NULL
              OR pct_independent_board IS NULL
            THEN NULL
            WHEN indep_board_lag1 - pct_independent_board >= 0.20
            THEN 1
            ELSE 0
        END AS flag_board_independence_drop,

        CASE
            WHEN has_three_consecutive_years = 1
            THEN STDDEV_SAMP(total_revenue) OVER (
                PARTITION BY ein
                ORDER BY tax_year
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            )
            ELSE NULL
        END AS rev_volatility_3yr,

        CASE
            WHEN has_three_consecutive_years = 1
            THEN AVG(net_income) OVER (
                PARTITION BY ein
                ORDER BY tax_year
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            )
            ELSE NULL
        END AS avg_net_income_3yr,

        CASE
            WHEN has_three_consecutive_years = 1
            THEN 3
            ELSE NULL
        END AS historical_filing_depth

    FROM validated
""")

longitudinal_export_path = (
    OUTPUT_DIR / "nccs_longitudinal_features_v1.parquet"
)

if WRITE_OUTPUTS:
    con.execute(f"""
        COPY nccs_longitudinal_features_v1
        TO '{longitudinal_export_path}'
        (FORMAT PARQUET, COMPRESSION 'SNAPPY')
    """)
    print("Longitudinal features saved.")
else:
    print("Longitudinal features export skipped.")

In [ ]:
columns = con.execute("""
    DESCRIBE SELECT * FROM nccs_header
""").df()

display(
    columns[
        columns["column_name"].str.contains(
            "DATE|TIME|AMEND|SUBMISSION|RETURN",
            case=False,
            na=False
        )
    ]
)

In [ ]:
display(con.execute("""
    SELECT
        EIN2,
        tax_year,
        RETURN_AMENDED_X,
        RETURN_TIME_STAMP,
        F9_00_IRS_SEC_SUBMISSION_DATE,
        TAX_PERIOD_END_DATE,
        OBJECTID
    FROM nccs_header
    WHERE EIN2 IN (
        SELECT EIN2
        FROM nccs_header
        GROUP BY EIN2, tax_year
        HAVING COUNT(*) > 1
    )
    ORDER BY EIN2, tax_year, RETURN_TIME_STAMP DESC
    LIMIT 25
""").df())

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*) AS total_filings,
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(RETURN_TIME_STAMP), '') IS NOT NULL
        ) AS has_return_timestamp,
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(F9_00_IRS_SEC_SUBMISSION_DATE), '') IS NOT NULL
        ) AS has_submission_date,
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(RETURN_AMENDED_X), '') IS NOT NULL
        ) AS has_amended_flag
    FROM nccs_header
""").df())

In [ ]:
display(con.execute("""
    WITH duplicate_groups AS (
        SELECT
            EIN2,
            tax_year,
            COUNT(*) AS filing_count,
            COUNT(DISTINCT RETURN_TIME_STAMP) AS distinct_timestamps,
            COUNT(DISTINCT RETURN_AMENDED_X) AS distinct_amendment_flags
        FROM nccs_header
        GROUP BY EIN2, tax_year
        HAVING COUNT(*) > 1
    )
    SELECT
        COUNT(*) AS duplicate_ein_year_groups,
        SUM(filing_count - 1) AS extra_filings,
        COUNT(*) FILTER (
            WHERE distinct_timestamps = 1
        ) AS groups_with_identical_timestamps,
        COUNT(*) FILTER (
            WHERE distinct_timestamps > 1
        ) AS groups_with_different_timestamps,
        COUNT(*) FILTER (
            WHERE distinct_amendment_flags > 1
        ) AS groups_with_mixed_amendment_flags
    FROM duplicate_groups
""").df())

In [ ]:
display(con.execute("""
    SELECT
        RETURN_AMENDED_X,
        COUNT(*) AS filings
    FROM nccs_header
    GROUP BY RETURN_AMENDED_X
    ORDER BY filings DESC
""").df())

## 9. Label and modeling coverage gate

**Important:** Missing disclosures are not automatically negative. This cell documents the label distribution and how much data complete-case modeling would exclude. Historical growth features use prior *available filings*, not necessarily consecutive years.

In [ ]:
display(con.execute("""SELECT tax_year, COUNT(*) rows,
COUNT(*) FILTER (WHERE fraud_label=1) positives,
COUNT(*) FILTER (WHERE fraud_label=0) negatives,
COUNT(*) FILTER (WHERE fraud_label IS NULL) unknowns
FROM nccs_longitudinal_features_v1 GROUP BY 1 ORDER BY 1""").df())
print('Only explicit 0/1 labels will be used in supervised training.')


## 10. Temporal split and supervised/unsupervised baselines

The next cells preserve the newer notebook’s model comparison and ranking workflow. Results may differ because missing labels are no longer silently treated as negatives.

In [ ]:
import pandas as pd
import numpy as np

# Define Temporal Split Strategy
train_years = list(range(2010, 2021))  # 2010-2020 (Training Window)
val_years   = [2021, 2022]             # 2021-2022 (Validation Window)
test_years  = [2023, 2024]             # 2023-2024 (Holdout Test Window)

print(f" Temporal Split Strategy:")
print(f" - Training Horizon:     {train_years[0]}–{train_years[-1]}")
print(f" - Validation Horizon:   {val_years[0]}–{val_years[-1]}")
print(f" - Holdout Test Horizon: {test_years[0]}–{test_years[-1]}")

# Record & Feature Distribution Across Splits
split_summary = con.execute(f"""
    SELECT 
        CASE 
            WHEN TRY_CAST(tax_year AS INTEGER) BETWEEN {train_years[0]} AND {train_years[-1]} THEN '1_Train (2010-2020)'
            WHEN TRY_CAST(tax_year AS INTEGER) IN ({','.join(map(str, val_years))}) THEN '2_Validation (2021-2022)'
            WHEN TRY_CAST(tax_year AS INTEGER) IN ({','.join(map(str, test_years))}) THEN '3_Test (2023-2024)'
            ELSE 'Other'
        END AS dataset_split,
        COUNT(*) AS record_count,
        COUNT(DISTINCT ein) AS unique_eins,
        SUM(fraud_label) AS positive_disclosures,
        ROUND(AVG(COALESCE(flag_sustained_deficit, 0)) * 100, 2) AS pct_sustained_deficit,
        ROUND(AVG(COALESCE(flag_small_board, 0)) * 100, 2) AS pct_small_board,
        ROUND(AVG(COALESCE(flag_board_independence_drop, 0)) * 100, 2) AS pct_board_indep_drop
    FROM nccs_longitudinal_features_v1
    GROUP BY 1
    ORDER BY 1 ASC;
""").df()

print("\n== Point-in-Time Record & Feature Distribution Across Splits ===")
display(split_summary)

# Entity Overlap & Disjointness Check
ein_overlap = con.execute(f"""
    WITH TrainEINs AS (
        SELECT DISTINCT ein FROM nccs_longitudinal_features_v1 
        WHERE TRY_CAST(tax_year AS INTEGER) BETWEEN {train_years[0]} AND {train_years[-1]}
    ),
    TestEINs AS (
        SELECT DISTINCT ein FROM nccs_longitudinal_features_v1 
        WHERE TRY_CAST(tax_year AS INTEGER) IN ({','.join(map(str, test_years))})
    )
    SELECT 
        (SELECT COUNT(*) FROM TestEINs) AS total_test_eins,
        COUNT(t.ein) AS returning_eins_in_test,
        (SELECT COUNT(*) FROM TestEINs) - COUNT(t.ein) AS new_unseen_eins_in_test,
        ROUND(COUNT(t.ein) * 100.0 / (SELECT COUNT(*) FROM TestEINs), 2) AS pct_returning_eins
    FROM TestEINs te
    LEFT JOIN TrainEINs t ON te.ein = t.ein;
""").df()

print("\n=== 🔒 Entity Overlap & Disjointness Analysis ===")
display(ein_overlap)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, IsolationForest
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score

# 1. Extract feature matrix from DuckDB
df_all = con.execute("SELECT * FROM nccs_longitudinal_features_v1;").df()

# Predictor columns
feature_cols = [
    'total_assets', 'total_liabilities', 'total_revenue', 'total_expenses',
    'net_income', 'debt_to_asset_ratio', 'voting_members', 'independent_members',
    'pct_independent_board', 'rev_growth_3yr', 'asset_growth_3yr',
    'rev_volatility_3yr', 'avg_net_income_3yr', 'flag_small_board',
    'flag_minority_independent_board', 'flag_board_independence_drop'
]

# Target column
target_col = 'fraud_label'

# Filter clean modeling subset
df_clean = df_all[df_all[target_col].isin([0,1])].dropna(subset=feature_cols).copy()

# Enforce point-in-time splits
df_train = df_clean[df_clean['tax_year'].astype(int).between(2010, 2020)]
df_val   = df_clean[df_clean['tax_year'].astype(int).between(2021, 2022)]
df_test  = df_clean[df_clean['tax_year'].astype(int).between(2023, 2024)]

X_train, y_train = df_train[feature_cols], df_train[target_col].astype(int)
X_val, y_val     = df_val[feature_cols], df_val[target_col].astype(int)
X_test, y_test   = df_test[feature_cols], df_test[target_col].astype(int)

# Note: X_train.shape[0] extracts the row count integer for formatting
print(f"📊 Modeling Splits -> Train: {X_train.shape[0]:,} rows ({y_train.sum()} positives) | "
      f"Val: {X_val.shape[0]:,} rows ({y_val.sum()} positives) | "
      f"Test: {X_test.shape[0]:,} rows ({y_test.sum()} positives)")

# 2. Capacity-Based Evaluator
def evaluate_capacity(y_true, scores, k_list=[250, 500, 1000]):
    precision, recall, _ = precision_recall_curve(y_true, scores)
    pr_auc_val = round(auc(recall, precision), 5)
    roc_auc_val = round(roc_auc_score(y_true, scores), 5)
    
    ranked_indices = np.argsort(scores)[::-1]
    results = {
        'PR-AUC': pr_auc_val,
        'ROC-AUC': roc_auc_val
    }
    
    for k in k_list:
        top_k_idx = ranked_indices[:k]
        found_at_k = int(y_true.iloc[top_k_idx].sum())
        results[f'Found@{k}'] = found_at_k
        results[f'Yield@{k} (%)'] = round((found_at_k / k) * 100, 2)
        
    return results

# 3. Fit Models & Compute Validation Performance
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'HistGradientBoosting': HistGradientBoostingClassifier(class_weight='balanced', random_state=42),
    'Isolation Forest': IsolationForest(contamination=0.05, random_state=42)
}

eval_results = []

for model_name, model in models.items():
    print(f"Training {model_name}...")
    if model_name == 'Isolation Forest':
        model.fit(X_train)
        val_scores = -model.decision_function(X_val)
    else:
        model.fit(X_train, y_train)
        val_scores = model.predict_proba(X_val)[:, 1]
        
    metrics = evaluate_capacity(y_val, val_scores, k_list=[250, 500, 1000])
    metrics['Model'] = model_name
    eval_results.append(metrics)

# 4. Display Validation Benchmark Summary
benchmark_df = pd.DataFrame(eval_results)[['Model', 'PR-AUC', 'ROC-AUC', 'Found@250', 'Yield@250 (%)', 'Found@500', 'Yield@500 (%)', 'Found@1000', 'Yield@1000 (%)']]
print("\n=== Validation Horizon Benchmark (2021–2022) ===")
display(benchmark_df)

In [ ]:
# Evaluate top-performing Logistic Regression model on Holdout Test Horizon (2023–2024)
test_scores = models['Logistic Regression'].predict_proba(X_test)[:, 1]
test_metrics = evaluate_capacity(y_test, test_scores, k_list=[100, 250, 500, 1000])

print("=== Holdout Test Horizon Benchmark (2023–2024) ===")
test_results_df = pd.DataFrame([test_metrics])
display(test_results_df)

# Export ranked test predictions with organization metadata for Databricks
df_test_ranked = df_test.copy()
df_test_ranked['risk_score'] = test_scores
df_test_ranked['risk_rank'] = df_test_ranked['risk_score'].rank(ascending=False, method='min').astype(int)

# Save ranked test candidate queue to data/processed/
ranked_export_path = OUTPUT_DIR / "nccs_investigation_queue_2023_2024.parquet"
if WRITE_OUTPUTS:
    df_test_ranked.to_parquet(ranked_export_path, index=False)
    print(f"Holdout test ranking queue exported to: {ranked_export_path.resolve()}")
else:
    print("Holdout test ranking queue export skipped.")

## 11. Next steps

1. Run **Notebook 08A** for detailed key, join, and source coverage checks. All 45 files existing does **not** guarantee join integrity.
2. Review the disclosure label value audit before interpreting classification metrics. If source codes differ from the explicit mappings, correct them and rerun.
3. Quantify the complete-case filtering loss and replace it with train-only imputation and missingness indicators in Notebook 09.
4. Verify whether filings are amended and refine the canonical filing selection rule.
5. Add peer-relative risk signals and compare current-year anomaly detection with supervised disclosure classification.
6. Treat investigation scores as review priorities, not fraud findings.
